# Tutorial 02: TokenLUT Redis — Disambiguation with RediSearch

This tutorial covers the **TokenLUT** (Token Lookup Table) system for disambiguation,
using RediSearch as the backend for fast indexed queries.

## What You'll Learn

1. **HLLSet Tensor View** — 2D (reg, zeros) representation
2. **TokenLUT Structure** — Mapping positions to candidate tokens
3. **RediSearch Integration** — Fast indexed queries
4. **Disambiguation Workflow** — From HLLSet to candidate tokens
5. **Triangulation** — Using n-gram layers for constraint filtering

## Prerequisites

- Redis with RediSearch module (redis-stack)
- HLLSet module loaded

In [6]:
import sys
sys.path.insert(0, '..')

import redis
from core.tokenlut_redis import TokenLUTRedis, TokenEntry
from core.hllset import HLLSet, DEFAULT_HASH_CONFIG

# Connect to Redis (podman redis-server with HLLSet + RediSearch)
# Use 127.0.0.1 explicitly (podman binds to IPv4 only)
r = redis.Redis(host='127.0.0.1', port=6379, decode_responses=True)

## 1. HLLSet Tensor View

An HLLSet can be viewed as a 2D tensor of shape (1024, 32):
- First dimension: register index (from hash prefix)
- Second dimension: zeros count (trailing zeros in hash)

Each token inscription sets a bit at position (reg, zeros).

> **Note**: We use `HLLSET.CREATEHASH` with Python-computed hashes to ensure
> positions match between Python's TokenLUT and Redis. The `HLLSET.CREATE` command
> uses Rust's MurmurHash3 internally, which differs from Python's MurmurHash64A.

In [13]:
# Helper: compute (reg, zeros, hash) for a token using Python hash
def hash_token(token: str):
    """Compute (reg, zeros, hash_full) using Python's murmur hash."""
    config = DEFAULT_HASH_CONFIG
    hash_full = config.hash(token)
    reg, zeros = config.hash_to_reg_zeros(token)
    return reg, zeros, hash_full

# Create an HLLSet using CREATEHASH (pre-computed hashes from Python)
tokens = ['apple', 'banana', 'cherry', 'date', 'elderberry']
hashes = [hash_token(t)[2] for t in tokens]

# Use CREATEHASH so positions match Python's hash function
key = r.execute_command('HLLSET.CREATEHASH', *hashes)
print(f'Created HLLSet: {key}')

# Get active positions (tensor view)
positions_flat = r.execute_command('HLLSET.POSITIONS', key)
positions = [(positions_flat[i], positions_flat[i+1]) 
             for i in range(0, len(positions_flat), 2)]

print(f'\nActive positions (reg, zeros):')
for reg, zeros in positions:
    print(f'  ({reg:4d}, {zeros})')

Created HLLSet: hllset:c3d7a428efa0e6003a2db926c1d3c5aa4fe2512d

Active positions (reg, zeros):
  ( 354, 0)
  ( 385, 1)
  ( 410, 2)
  ( 852, 2)
  ( 912, 3)


In [14]:
# Examine bit counts (c_s values for Horvitz-Thompson)
bitcounts = r.execute_command('HLLSET.BITCOUNTS', key)
print('Bit counts c_s (registers with bit s set):')
for s, count in enumerate(bitcounts[:8]):
    if count > 0:
        print(f'  c_{s} = {count}')

Bit counts c_s (registers with bit s set):
  c_0 = 1
  c_1 = 1
  c_2 = 2
  c_3 = 1


## 2. TokenLUT — The Disambiguation Table

TokenLUT stores the mapping: `(reg, zeros) → [candidate tokens]`

When we have an HLLSet fingerprint, we can:
1. Get all active positions
2. Look up candidate tokens for each position
3. Apply triangulation to narrow down

RediSearch provides fast indexed queries on this structure.

In [15]:
# Initialize TokenLUT with RediSearch
lut = TokenLUTRedis(r, index_name='demo:tokenlut', prefix='demo:entry:')

# Create the search index (one-time setup)
lut.create_index(drop_existing=True)
print(f'Created TokenLUT index: {lut.index_name}')

Created TokenLUT index: demo:tokenlut


In [16]:
# Verify Python hash gives positions matching the HLLSet
print('Token hashes and positions:')
for token in tokens:
    reg, zeros, h = hash_token(token)
    print(f'{token:12} → reg={reg:4d}, zeros={zeros}, hash={h}')

Token hashes and positions:
apple        → reg= 385, zeros=1, hash=10616410365961357697
banana       → reg= 852, zeros=2, hash=18066328439954051924
cherry       → reg= 354, zeros=0, hash=17144606365356158306
date         → reg= 912, zeros=3, hash=233091553539433360
elderberry   → reg= 410, zeros=2, hash=5790570204009214362


In [17]:
# Populate the LUT with our tokens
for token in tokens:
    reg, zeros, hash_full = hash_token(token)
    lut.add_token(
        token=token,
        reg=reg,
        zeros=zeros,
        hash_full=hash_full,
        layer=0  # unigrams
    )

print(f'Added {len(tokens)} tokens to LUT')
print(f'LUT stats: {lut.stats()}')

Added 5 tokens to LUT
LUT stats: {'total_tokens': 5, 'unique_positions': 5, 'positions_with_collisions': 0, 'max_collision_size': 1, 'index_name': 'demo:tokenlut', 'index_size_mb': 0.0001392364501953125}


## 3. Disambiguation Lookup

Given an HLLSet, we can recover candidate tokens by:
1. Getting active positions from the HLLSet
2. Looking up each position in the TokenLUT

Multiple tokens may map to the same position (hash collisions),
so we get a list of *candidates* per position.

In [18]:
# Disambiguation workflow
print('=== Disambiguation Workflow ===')
print(f'\n1. HLLSet key: {key}')
print(f'2. Active positions: {len(positions)}')

print('\n3. Lookup candidates for each position:')
for reg, zeros in positions:
    candidates = lut.lookup(reg, zeros)
    if candidates:
        tokens_str = ', '.join(c.token for c in candidates)
        print(f'   ({reg:4d}, {zeros}) → [{tokens_str}]')
    else:
        print(f'   ({reg:4d}, {zeros}) → [no candidates in LUT]')

=== Disambiguation Workflow ===

1. HLLSet key: hllset:c3d7a428efa0e6003a2db926c1d3c5aa4fe2512d
2. Active positions: 5

3. Lookup candidates for each position:
   ( 354, 0) → [cherry]
   ( 385, 1) → [apple]
   ( 410, 2) → [elderberry]
   ( 852, 2) → [banana]
   ( 912, 3) → [date]


## 4. Adding More Tokens and Collisions

Let's add more tokens to see how collisions are handled.

In [19]:
# Add a larger vocabulary
vocabulary = [
    'the', 'quick', 'brown', 'fox', 'jumps', 'over', 'lazy', 'dog',
    'hello', 'world', 'python', 'redis', 'search', 'index', 'query',
    'token', 'lookup', 'table', 'disambiguation', 'triangulation'
]

for token in vocabulary:
    reg, zeros, hash_full = hash_token(token)
    lut.add_token(token=token, reg=reg, zeros=zeros, 
                  hash_full=hash_full, layer=0)

print(f'Total entries: {lut.count()}')
print(f'Stats: {lut.stats()}')

Total entries: 25
Stats: {'total_tokens': 25, 'unique_positions': 25, 'positions_with_collisions': 0, 'max_collision_size': 1, 'index_name': 'demo:tokenlut', 'index_size_mb': 0.000652313232421875}


In [20]:
# Check for positions with multiple candidates (collisions)
positions = lut.active_positions()
collisions = []

for pos in positions:
    candidates = lut.lookup(pos[0], pos[1])
    if len(candidates) > 1:
        collisions.append((pos, candidates))

print(f'Positions with collisions: {len(collisions)}')
for pos, candidates in collisions:
    print(f'  {pos}: {[c.token for c in candidates]}')

Positions with collisions: 0


## 5. N-gram Layers and Triangulation

For disambiguation, we use multiple n-gram layers:
- **Layer 0**: Unigrams (single tokens)
- **Layer 1**: Bigrams (token pairs)
- **Layer 2**: Trigrams (token triples)

The `first_token` field links n-grams back to their starting unigram,
enabling triangulation constraints.

In [21]:
# Add some bigrams
bigrams = [
    'quick brown', 'brown fox', 'fox jumps', 'jumps over',
    'lazy dog', 'hello world', 'redis search'
]

for bigram in bigrams:
    reg, zeros, hash_full = hash_token(bigram)
    first_token = bigram.split()[0]
    lut.add_token(
        token=bigram,
        reg=reg,
        zeros=zeros,
        hash_full=hash_full,
        layer=1,
        first_token=first_token
    )

print(f'Added {len(bigrams)} bigrams')
print(f'Total entries: {lut.count()}')

Added 7 bigrams
Total entries: 32


In [22]:
# Query by layer
unigrams = lut.lookup_layer(0)
bigrams_found = lut.lookup_layer(1)

print(f'Unigrams (layer 0): {len(unigrams)}')
print(f'Bigrams (layer 1): {len(bigrams_found)}')

print('\nBigrams:')
for entry in bigrams_found:
    print(f'  "{entry.token}" (first_token="{entry.first_token}")')

Unigrams (layer 0): 25
Bigrams (layer 1): 7

Bigrams:
  "quick brown" (first_token="quick")
  "brown fox" (first_token="brown")
  "fox jumps" (first_token="fox")
  "jumps over" (first_token="jumps")
  "lazy dog" (first_token="lazy")
  "hello world" (first_token="hello")
  "redis search" (first_token="redis")


In [23]:
# Triangulation: find bigrams starting with a specific token
first_token = 'quick'
matching = lut.lookup_by_first_token(first_token, layer=1)

print(f'Bigrams starting with "{first_token}":')
for entry in matching:
    print(f'  "{entry.token}"')

Bigrams starting with "quick":
  "quick brown"


## 6. Full Disambiguation Example

Let's demonstrate the full workflow:
1. Create HLLSet from text
2. Get active positions
3. Lookup candidates from LUT
4. Apply constraints to narrow down

In [24]:
# Create HLLSet from a phrase (both unigrams and bigrams)
phrase_tokens = ['quick', 'brown', 'fox', 'quick brown', 'brown fox']
phrase_hashes = [hash_token(t)[2] for t in phrase_tokens]

# Use CREATEHASH for consistent positions
phrase_key = r.execute_command('HLLSET.CREATEHASH', *phrase_hashes)

print(f'Created phrase HLLSet: {phrase_key}')
print(f'Cardinality: {r.execute_command("HLLSET.CARD", phrase_key)}')

# Get positions
pos_flat = r.execute_command('HLLSET.POSITIONS', phrase_key)
phrase_positions = [(pos_flat[i], pos_flat[i+1]) 
                    for i in range(0, len(pos_flat), 2)]

print(f'\nActive positions: {len(phrase_positions)}')

Created phrase HLLSet: hllset:741994f2262e8e11fb50a9bf21ae430c08e67855
Cardinality: 5

Active positions: 5


In [25]:
# Lookup all candidates
candidates_by_pos = lut.lookup_positions(phrase_positions)

print('Candidates per position:')
for pos, candidates in candidates_by_pos.items():
    if candidates:
        for c in candidates:
            layer_str = 'unigram' if c.layer == 0 else f'{c.layer}-gram'
            print(f'  {pos} → "{c.token}" ({layer_str})')

Candidates per position:
  (193, 1) → "quick brown" (1-gram)
  (573, 0) → "brown" (unigram)
  (678, 3) → "fox" (unigram)
  (892, 2) → "quick" (unigram)
  (954, 2) → "brown fox" (1-gram)


In [26]:
# Apply triangulation: filter unigrams based on bigram constraints
print('\n=== Triangulation ===')

# Get all unigram candidates
unigram_candidates = set()
for pos, candidates in candidates_by_pos.items():
    for c in candidates:
        if c.layer == 0:
            unigram_candidates.add(c.token)

print(f'All unigram candidates: {unigram_candidates}')

# Get first_tokens from bigrams (constraint)
bigram_first_tokens = set()
for pos, candidates in candidates_by_pos.items():
    for c in candidates:
        if c.layer == 1:
            bigram_first_tokens.add(c.first_token)
            
print(f'Bigram first_tokens: {bigram_first_tokens}')

# Filter: unigrams must appear as first_token of some bigram
filtered = unigram_candidates & bigram_first_tokens
print(f'Filtered unigrams (appear in bigrams): {filtered}')


=== Triangulation ===
All unigram candidates: {'quick', 'fox', 'brown'}
Bigram first_tokens: {'quick', 'brown'}
Filtered unigrams (appear in bigrams): {'quick', 'brown'}


## 7. Streaming Ingestion with Redis Streams

For large vocabularies, we use **Redis Streams** for batch ingestion:
- **Producer**: Submits tokens to stream (no round-trips per token)
- **Consumer**: Processes stream, computes hashes, stores in RediSearch
- **Backpressure**: Stream handles rate mismatches

This eliminates extra round-trips and enables pipeline-style processing.

In [28]:
from core.tokenlut_stream import TokenLUTStream, create_stream_lut, StreamConfig

# Create a streaming LUT (separate from our demo LUT)
stream_lut = create_stream_lut(
    r, 
    index_name='stream:tokenlut', 
    prefix='stream:entry:',
    stream_key='stream:tokens'
)
print(f'Created streaming LUT: {stream_lut.index_name}')
print(f'Stream key: {stream_lut.config.stream_key}')

Created streaming LUT: stream:tokenlut
Stream key: stream:tokens


In [29]:
# Batch ingest unigrams via stream
stream_vocab = [
    'alpha', 'beta', 'gamma', 'delta', 'epsilon',
    'zeta', 'eta', 'theta', 'iota', 'kappa'
]

# Submit to stream (fast - no per-token round trips)
stream_lut.ingest_tokens(stream_vocab, layer=0)
print(f'Submitted {stream_lut.stats.tokens_submitted} tokens to stream')
print(f'Stream length: {stream_lut.stream_length()}')

Submitted 10 tokens to stream
Stream length: 10


In [30]:
# Add bigrams with first_token via stream
stream_bigrams = [
    ('alpha beta', 'alpha'),
    ('beta gamma', 'beta'),
    ('gamma delta', 'gamma'),
    ('delta epsilon', 'delta'),
]

stream_lut.ingest_ngrams(stream_bigrams, layer=1)
print(f'Total submitted: {stream_lut.stats.tokens_submitted}')
print(f'Stream length: {stream_lut.stream_length()}')

Total submitted: 14
Stream length: 14


In [31]:
# Process the stream (consumer side - computes hashes, stores in RediSearch)
processed = stream_lut.process_pending()
print(f'Processed {processed} entries from stream')
print(f'Stream length after processing: {stream_lut.stream_length()}')
print(f'\nIngestion stats: {stream_lut.get_stats()}')

Processed 14 entries from stream
Stream length after processing: 14

Ingestion stats: {'tokens_submitted': 14, 'tokens_processed': 14, 'tokens_failed': 0, 'batches_submitted': 2, 'batches_processed': 1, 'elapsed_seconds': 19.52, 'tokens_per_second': 0.7}


In [32]:
# Verify: query the stream LUT
print('Entries in stream LUT:')
print(f'  Total count: {stream_lut.count()}')

# Query bigrams
from redis.commands.search.query import Query
results = r.ft('stream:tokenlut').search(Query('@layer:[1 1]'))
print(f'\nBigrams from stream:')
for doc in results.docs:
    print(f'  "{doc.token}" (first_token="{doc.first_token}")')

Entries in stream LUT:
  Total count: 14

Bigrams from stream:
  "alpha beta" (first_token="alpha")
  "beta gamma" (first_token="beta")
  "gamma delta" (first_token="gamma")
  "delta epsilon" (first_token="delta")


In [33]:
# Direct pipeline mode (bypass stream for small batches)
direct_tokens = ['omega', 'psi', 'chi', 'phi']
count = stream_lut.add_tokens_pipeline(direct_tokens, layer=0)
print(f'Added {count} tokens via direct pipeline')
print(f'Total in LUT: {stream_lut.count()}')

Added 4 tokens via direct pipeline
Total in LUT: 18


In [34]:
# Cleanup streaming LUT
stream_lut.drop_index(delete_documents=True)
stream_lut.delete_stream()
print('Stream LUT cleaned up')

Stream LUT cleaned up


## 8. Cleanup

In [27]:
# Clean up
lut.drop_index(delete_documents=True)
r.execute_command('HLLSET.DEL', key)
r.execute_command('HLLSET.DEL', phrase_key)
print('Cleaned up')

Cleaned up


## Summary

In this tutorial, you learned:

1. **Tensor View**: HLLSet as 2D (reg, zeros) positions via `HLLSET.POSITIONS`
2. **TokenLUT**: RediSearch-backed lookup table for `(reg, zeros) → tokens`
3. **Disambiguation**: Getting candidate tokens from active positions
4. **Triangulation**: Using n-gram layers to constrain candidates
5. **Streaming Ingestion**: Batch token ingestion via Redis Streams

### Key Commands

| Command | Description |
|---------|-------------|
| `HLLSET.POSITIONS key` | Get all (reg, zeros) pairs |
| `HLLSET.POPCOUNT key` | Total set bits |
| `HLLSET.BITCOUNTS key` | c_s counts per bit position |
| `HLLSET.REGISTER key reg` | Get register bitmap |
| `HLLSET.HASBIT key reg zeros` | Check specific bit |

### Streaming API

```python
from core.tokenlut_stream import create_stream_lut

stream = create_stream_lut(r, index_name='vocab:lut')
stream.ingest_tokens(tokens, layer=0)      # Submit to stream
stream.ingest_ngrams(bigrams, layer=1)     # N-grams with first_token
stream.process_pending()                    # Process stream → RediSearch
```

### Next Steps

- **Tutorial 03**: HLL Lattice Redis — Content-addressed DAG
- **Tutorial 07**: Disambiguation Redis — Full token recovery